# Zöllner Binary Molecule: Four-Body Bound State — Live ($\eta = 0.8$, $a = 0.5$)

This notebook demonstrates a **bound four-body state** in Weber
electrodynamics with a **strong Zöllner extension**: two sub-critical
like-charge pairs orbit each other under Zöllner-enhanced unlike-charge
attraction, forming a **binary molecule**.

## Physics Background

### Weber's Velocity-Dependent Force

Weber's force law between two charges $q_i$, $q_j$ separated by distance
$r$ with radial velocity $\dot{r}$ and acceleration $\ddot{r}$ is

$$F = \frac{q_i q_j}{r^2}\left(1 - \frac{\dot{r}^2}{2c^2} + \frac{r\ddot{r}}{c^2}\right)$$

The velocity- and acceleration-dependent terms create an **effective
inertial mass** that depends on separation:

$$\mu_{\text{eff}}(r) = \mu\left(1 - \frac{\rho}{r}\right)$$

where $\mu$ is the reduced mass and $\rho = q_i q_j / (\mu c^2)$ is
the **critical radius**.

### Critical Radius and Sub-Critical Binding

For two like charges ($q_i q_j > 0$), the critical radius $\rho > 0$ is
a real positive distance. The effective inertial mass changes sign at
$r = \rho$, creating two permanently separated dynamical regimes:

- **Distant state** ($r > \rho$): ordinary Coulomb-like repulsion and
  scattering.
- **Molecular state** ($r < \rho$): the particles are bound, oscillating
  between their initial separation $r_0$ and $r = 0$. The sign reversal
  of $\mu_{\text{eff}}$ turns repulsion into effective attraction.

No continuous trajectory can cross $r = \rho$. Weber called this a
"molecular movement" (Sixth Memoir, §9.9).

### From Planetary Atom to Binary Molecule

The three-body **planetary atom** combines one sub-critical nucleus
(two like charges) with a single orbiting unlike charge. The **binary
molecule** generalizes this to four bodies: *two* sub-critical pairs,
each acting as a composite "nucleus," orbiting each other under mutual
Coulomb attraction.

| Particle | Charge | Pair | Role |
|---|---|---|---|
| 1 | $+q$ | A | Sub-critical positive nucleus |
| 2 | $+q$ | A | Sub-critical positive nucleus |
| 3 | $-q$ | B | Sub-critical negative nucleus |
| 4 | $-q$ | B | Sub-critical negative nucleus |

Each pair is bound in the molecular state ($r_{\text{nuc}} < \rho$),
oscillating radially at period
$T_{\text{nuc}} \approx 2\sqrt{2}\,r_{\text{nuc}}/c$. The net charges
of the two pairs ($+2q$ and $-2q$) produce an attractive inter-pair
force, enabling a stable binary orbit at separation $R \gg r_{\text{nuc}}$.

The total system charge is **zero** — a charge-neutral bound state held
together entirely by Weber's velocity-dependent dynamics.

### The Zöllner Extension

Zöllner's electrogravitational theory (1872) postulates that unlike-charge
attraction is slightly stronger than like-charge repulsion by a mismatch
parameter $a > 0$. Each pair receives a coupling factor $\kappa_{ij}$:

$$\kappa_{ij} = \begin{cases} 1 + a & \text{if } q_i q_j < 0 \text{ (unlike charges)} \\ 1 & \text{if } q_i q_j > 0 \text{ (like charges)} \end{cases}$$

For the binary molecule, 4 of 6 pairs are unlike-charge (enhanced),
while the 2 intra-pair interactions are unaffected:

| Pair | Charges | $\kappa$ |
|---|---|---|
| (1,2) | $+,+$ | $1.0$ |
| (1,3) | $+,-$ | $1 + a$ |
| (1,4) | $+,-$ | $1 + a$ |
| (2,3) | $+,-$ | $1 + a$ |
| (2,4) | $+,-$ | $1 + a$ |
| (3,4) | $-,-$ | $1.0$ |

The Zöllner enhancement selectively strengthens the inter-pair orbital
attraction without altering the internal nuclear oscillation of either
pair. This is the same circularization mechanism seen in the three-body
planetary atom, now applied to a symmetric binary orbit.

### Binary Orbit Dynamics

When $R \gg r_{\text{nuc}}$, each pair appears as a point charge to the
other. The four cross-pair Coulomb interactions sum to an effective
attractive force:

$$F_{\text{binary}} \approx \frac{4\kappa q^2}{R^2}$$

where $\kappa = 1 + a$ for the Zöllner-enhanced case. The binary
reduced mass is $\mu_{\text{bin}} = M_{\text{pair}}^2 / (2 M_{\text{pair}}) = m$
(for equal particle masses $m$), giving a circular orbit speed:

$$v_{\text{circ}} = \sqrt{\frac{4q^2}{\mu_{\text{bin}} R}}$$

The timescale separation
$T_{\text{orb}} / T_{\text{nuc}} \approx 89$ ensures the fast nuclear
oscillations average out on the orbital timescale, maintaining stability.

### This Run

This notebook uses the **live streaming** animation viewer, which
integrates the system in real time without precomputing the full
trajectory. Use the speed slider to control the integration rate.

At $\eta = 0.8$ with $a = 0.5$, the Zöllner coupling strengthens all
four inter-pair interactions by 50%. The velocity parameter $\eta$
sets the initial orbital speed to 80% of the (bare) circular speed.
The Zöllner enhancement circularizes the resulting binary orbit,
analogous to the strong circularization effect observed in the
three-body planetary atom at the same parameters.

**References**: Weber, Sixth Memoir (1871) §§9.8--9.17; Zöllner (1872);
Frauenfelder & Weber, *Anal. Math. Phys.* **14**:31 (2024); see
`research/theory/ZollnerElectrogravitationalTheory.md`,
`research/theory/InitialConditions.md`, and
`research/investigations/ThreeBodyBoundStates.md`.

In [ ]:
using WeberElectrodynamics
using LinearAlgebra
using Printf
using GLMakie  # or CairoMakie, WGLMakie

## 1. System Construction and Physical Parameters

In [ ]:
# Physical parameters
m = 1.0               # equal masses
q_pos = 1.0           # positive charges (pair A)
q_neg = -1.0          # negative charges (pair B)
c = 4.0

# Derived quantities — nuclear binding (identical for both pairs)
mu_nuc = m * m / (m + m)          # nucleus reduced mass = 0.5
rho = q_pos^2 / (mu_nuc * c^2)   # critical radius = 0.125

# Nucleus parameters (both pairs)
r_nuc = 0.05   # sub-critical internal separation
T_nuc = 2 * sqrt(2) * r_nuc / c   # nuclear oscillation period

# Binary orbit parameters
M_pair = 2m                                           # mass of each composite pair
mu_binary = M_pair * M_pair / (M_pair + M_pair)       # binary reduced mass = 1.0
R = 1.0                                                # COM-to-COM separation
Q_eff_binary = 4 * q_pos^2                            # |Q_A * Q_B| = |2q * (-2q)| = 4q^2
v_circ_rel = sqrt(Q_eff_binary / (mu_binary * R))     # relative circular speed = 2.0
T_orb = 2 * pi * R / v_circ_rel                       # orbital period

# Integration parameters
dt = 1e-4
bounce_r = 0.02

system = HamiltonianSystem(4, 2)

@printf("Four-body binary molecule:\n")
@printf("  Particles:  %d (2D)\n", system.n_particles)
@printf("  DOF:        %d\n", system.degrees_of_freedom)
@printf("\nPair A — positive nucleus (particles 1, 2):\n")
@printf("  q1 = q2 = +%.1f, m = %.1f\n", q_pos, m)
@printf("  r_nuc = %.4f < rho = %.4f\n", r_nuc, rho)
@printf("  T_nuc ~ %.6f\n", T_nuc)
@printf("\nPair B — negative nucleus (particles 3, 4):\n")
@printf("  q3 = q4 = %.1f, m = %.1f\n", q_neg, m)
@printf("  r_nuc = %.4f < rho = %.4f\n", r_nuc, rho)
@printf("  T_nuc ~ %.6f\n", T_nuc)
@printf("\nBinary orbit (pair A <-> pair B):\n")
@printf("  R = %.2f, mu_binary = %.4f\n", R, mu_binary)
@printf("  v_circ_rel = %.4f, T_orb ~ %.4f\n", v_circ_rel, T_orb)
@printf("  T_orb / T_nuc = %.1f  (timescale separation)\n", T_orb / T_nuc)
@printf("\nIntegration:\n")
@printf("  dt = %.0e, bounce_r = %.2f, c = %.1f\n", dt, bounce_r, c)

## 2. Initial Conditions and Problem Setup

In [ ]:
function make_binary_molecule_ic(r_nuc, R, eta_orb, m, q_pos, q_neg, c)
    # Pair A (positive) centered at (0, -R/2), internal axis along x
    x1 = -r_nuc / 2;  y1 = -R / 2
    x2 = +r_nuc / 2;  y2 = -R / 2

    # Pair B (negative) centered at (0, +R/2), internal axis along x
    x3 = -r_nuc / 2;  y3 = +R / 2
    x4 = +r_nuc / 2;  y4 = +R / 2

    # Binary orbit: two composite bodies of mass 2m each
    M_pair = 2m
    mu_binary = M_pair * M_pair / (M_pair + M_pair)
    Q_eff = 4 * q_pos^2
    v_circ_rel = sqrt(Q_eff / (mu_binary * R))
    v_orb_rel = eta_orb * v_circ_rel
    v_each = v_orb_rel / 2

    # Pair A at bottom moves right (+x), Pair B at top moves left (-x)
    # -> counterclockwise binary orbit
    # Both particles in a pair share the pair's COM velocity (no internal motion)
    px1 = m * v_each;   py1 = 0.0
    px2 = m * v_each;   py2 = 0.0
    px3 = -m * v_each;  py3 = 0.0
    px4 = -m * v_each;  py4 = 0.0

    # Shift to center-of-mass frame (already at origin by symmetry)
    M_total = 4m
    cx = (m * x1 + m * x2 + m * x3 + m * x4) / M_total
    cy = (m * y1 + m * y2 + m * y3 + m * y4) / M_total

    q0 = [x1 - cx, y1 - cy, x2 - cx, y2 - cy, x3 - cx, y3 - cy, x4 - cx, y4 - cy]
    p0 = [px1, py1, px2, py2, px3, py3, px4, py4]
    return q0, p0, v_circ_rel, v_orb_rel
end

eta = 0.8
a = 0.5
tmax = Inf

q0, p0, vc_rel, vorb_rel = make_binary_molecule_ic(r_nuc, R, eta, m, q_pos, q_neg, c)

prob = HamiltonianProblem(system, (0.0, tmax), q0, p0;
    masses = [m, m, m, m], charges = [q_pos, q_pos, q_neg, q_neg], c = c, dt = dt,
    zollner=ZollnerOptions(enabled=true, a=a))

@printf("Binary molecule: eta = %.1f, a = %.1f\n", eta, a)
@printf("  v_orb_rel = %.4f (v_circ_rel = %.4f)\n", vorb_rel, vc_rel)
@printf("  v_each    = %.4f\n", vorb_rel / 2)
@printf("\nZollner kappas:\n")
pair_labels = ["(1,2) +,+", "(1,3) +,-", "(1,4) +,-",
               "(2,3) +,-", "(2,4) +,-", "(3,4) -,-"]
for (k, label) in enumerate(pair_labels)
    @printf("  kappa_%s = %.2f\n", label, kappas(prob)[k])
end

## 3. Live Animation

The streaming animation viewer integrates the system in real time,
displaying rolling trajectories, energy, momentum, angular momentum,
and phase space. Use the **Speed** slider to control how many integration
steps are computed per frame.

In [ ]:
animate_weber(prob; buffer_size = 2000, tail_length = 200, compute_batch = 1)